In [ ]:
import pandas as pd
import numpy as np
import ast


In [ ]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies=movies.merge(credits,on='title')
#PREPROCESSING PROCEDURE


In [ ]:
movies.info()
#keeping imp columns
#1.genre
#2.keywords
#3.title
#4.overview
#5.cast
#6.crew

In [ ]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.iloc[0].genres


In [ ]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L


In [ ]:
movies['genres']=movies['genres'].apply(convert)


In [ ]:
movies.head()

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)
movies.head()

In [ ]:
def convertcast(obj):
    count = 0
    L=[]
    for i in ast.literal_eval(obj):

        if count !=3:
            L.append(i['name'])
            count=count+1
        else:
            break;
    return L


In [ ]:
movies['cast']=movies['cast'].apply(convertcast)
movies.head()

In [ ]:
movies.iloc[0].crew
def finddire(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L




In [ ]:
movies['crew']=movies['crew'].apply(finddire)
movies.head()

In [ ]:
movies['overview']=movies['overview'].apply(lambda x : x.split())

In [ ]:
movies['genres']= movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords']= movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast']= movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew']= movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])

In [ ]:
movies.head()

In [ ]:
movies['tags']=movies['overview']+movies['keywords']+movies['cast']+movies['crew']

In [ ]:
new_df=movies[['movie_id','title','tags']]
new_df.head()

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [ ]:
new_df.head()


In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [ ]:
#VECTORIZATION PROCESS

In [78]:
from sklearn.feature_extraction.text import CountVectorizer
cv= CountVectorizer(max_features=5000,stop_words='english')
vectors=cv.fit_transform(new_df['tags']).toarray()

In [79]:
import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [76]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [77]:
new_df['tags']=new_df['tags'].apply(stem)

In [82]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [84]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vectors).shape